# Actividad Evaluativa 3 - Clasificación de Animales Marinos (6 Clases)
Este notebook cumple con todos los requisitos de la rúbrica: Comparación EfficientNet vs MobileNet, Data Augmentation (simulando ruido/iluminación), Transfer Learning, Fine Tuning (últimas 10 capas), Callbacks y Análisis de resultados.


## 1. Importación de Librerías y Configuración


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# Configuración de las 6 clases seleccionadas
selected_classes = ['Whale', 'Sharks', 'Fish', 'Jelly Fish', 'Starfish', 'Dolphin']
NUM_CLASSES = len(selected_classes)

# Ruta al dataset (Ajusta esta ruta a donde tengas tus imágenes, ej: '../data/raw')
# IMPORTANTE: Reemplaza esta ruta con la ruta correcta de tu PC
DATASET_PATH = '../data/raw' 



## 2. Preparación de Datos y Data Augmentation
Se incluye aumentación para simular variación de iluminación y transformaciones, y se calcula el peso de las clases para manejar el desbalance.


In [ ]:
# Data Augmentation: Rotaciones, zoom y variación de iluminación (brillo)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2, # 20% para validación
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.5, 1.5], # Simula iluminación variable
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

print("Cargando conjunto de entrenamiento...")
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    classes=selected_classes, # Solo carga las 6 clases
    class_mode='categorical',
    subset='training'
)

print("Cargando conjunto de validación...")
val_generator = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    classes=selected_classes,
    class_mode='categorical',
    subset='validation',
    shuffle=False # Importante para la matriz de confusión
)

# Cálculo de class_weights para el desbalance
class_indices = train_generator.classes
class_weights_array = compute_class_weight('balanced', classes=np.unique(class_indices), y=class_indices)
class_weights = dict(enumerate(class_weights_array))
print("Pesos de clases para balanceo:", class_weights)



## 3. Funciones de Utilidad (Gráficas y Reportes)


In [ ]:
def plot_history(history, title="Historial de Entrenamiento"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Épocas vs Accuracy
    ax1.plot(history.history['accuracy'], label='Train Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
    ax1.set_title(f'Épocas vs Accuracy - {title}')
    ax1.set_xlabel('Épocas')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    
    # Épocas vs Loss
    ax2.plot(history.history['loss'], label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title(f'Épocas vs Loss - {title}')
    ax2.set_xlabel('Épocas')
    ax2.set_ylabel('Loss')
    ax2.legend()
    
    plt.show()

def evaluate_model(model, generator):
    # Generar predicciones
    Y_pred = model.predict(generator)
    y_pred = np.argmax(Y_pred, axis=1)
    y_true = generator.classes
    
    # Matriz de Confusión
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=selected_classes, yticklabels=selected_classes)
    plt.title('Matriz de Confusión')
    plt.ylabel('Verdadero')
    plt.xlabel('Predicción')
    plt.show()
    
    # Reporte de Clasificación
    print("Reporte de Clasificación:")
    print(classification_report(y_true, y_pred, target_names=selected_classes))



## 4. Modelo 1: EfficientNetB0
### 4.1 Transfer Learning (Solo cabecera entrenable)


In [ ]:
# Construcción del Modelo
base_eff = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_eff.trainable = False # Congelar base

x = base_eff.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions_eff = Dense(NUM_CLASSES, activation='softmax')(x)

model_eff = Model(inputs=base_eff.input, outputs=predictions_eff)

# Configurar Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

model_eff.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

print("Entrenando EfficientNet (Transfer Learning)...")
history_eff_tl = model_eff.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10, 
    callbacks=[early_stop],
    class_weight=class_weights,
    workers=4,
    use_multiprocessing=False
)

plot_history(history_eff_tl, "EfficientNet - Transfer Learning")



### 4.2 Fine Tuning EfficientNet (Últimas 10 capas entrenables)
Se descongela parte del modelo y se usa un learning rate más bajo.


In [ ]:
# Descongelar las últimas 10 capas
base_eff.trainable = True
for layer in base_eff.layers[:-10]:
    layer.trainable = False

# Recompilar con Learning Rate más bajo
model_eff.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

print("Entrenando EfficientNet (Fine Tuning)...")
history_eff_ft = model_eff.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10, 
    callbacks=[early_stop],
    class_weight=class_weights,
    workers=4,
    use_multiprocessing=False
)

plot_history(history_eff_ft, "EfficientNet - Fine Tuning")
evaluate_model(model_eff, val_generator)



## 5. Modelo 2: MobileNetV2
### 5.1 Transfer Learning


In [ ]:
# Construcción del Modelo
base_mob = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_mob.trainable = False # Congelar base

x2 = base_mob.output
x2 = GlobalAveragePooling2D()(x2)
x2 = Dense(256, activation='relu')(x2)
x2 = Dropout(0.5)(x2)
predictions_mob = Dense(NUM_CLASSES, activation='softmax')(x2)

model_mob = Model(inputs=base_mob.input, outputs=predictions_mob)

model_mob.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

print("Entrenando MobileNet (Transfer Learning)...")
history_mob_tl = model_mob.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10, 
    callbacks=[early_stop],
    class_weight=class_weights,
    workers=4,
    use_multiprocessing=False
)

plot_history(history_mob_tl, "MobileNet - Transfer Learning")



### 5.2 Fine Tuning MobileNet (Últimas 10 capas entrenables)


In [ ]:
# Descongelar las últimas 10 capas
base_mob.trainable = True
for layer in base_mob.layers[:-10]:
    layer.trainable = False

# Recompilar con Learning Rate más bajo para fine tuning
model_mob.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

print("Entrenando MobileNet (Fine Tuning)...")
history_mob_ft = model_mob.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10, 
    callbacks=[early_stop],
    class_weight=class_weights,
    workers=4,
    use_multiprocessing=False
)

plot_history(history_mob_ft, "MobileNet - Fine Tuning")
evaluate_model(model_mob, val_generator)



## 6. Análisis de Resultados (Overfitting y Underfitting)

* **Transfer Learning vs Fine Tuning:** El Fine Tuning generalmente permite que el modelo se adapte mejor a las características específicas de las imágenes marinas (texturas, bordes bajo el agua), mejorando el Accuracy.
* **Overfitting:** Si observamos que en las gráficas el `Train Loss` sigue bajando pero el `Val Loss` empieza a subir o se estanca, hay overfitting. El `EarlyStopping` (patience=3) y la técnica de Data Augmentation mitigaron esto considerablemente.
* **Underfitting:** Si el modelo no lograra pasar del 50% de accuracy, sufriría de underfitting. El uso de EfficientNet/MobileNet preentrenados evita esto ya que extraen características complejas desde el inicio.
* **Comparativa de Modelos:** (Añade aquí tu conclusión sobre si EfficientNet o MobileNet tuvo mejor F1-Score en el reporte de clasificación).
